In [ ]:
pip install deltalake

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BUCKET, STORAGE_OPTIONS, info
info()

BRONZE = f's3://{BUCKET}/bronze/vatsim'
BRONZE_TEST = f's3://{BUCKET}/bronze/_test_vatsim'
print('Destino bronze:', BRONZE)

In [ ]:
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError


def leer_watermark(destino: str = BRONZE) -> str | None:
    """Devuelve el máximo 'snapshot_ts' ya presente en bronze."""
    try:
        dt = DeltaTable(destino, storage_options=STORAGE_OPTIONS)
    except TableNotFoundError:
        return None
    ts = dt.to_pandas(columns=["snapshot_ts"])["snapshot_ts"].dropna()
    return None if ts.empty else str(ts.max())

In [ ]:
import requests

r = requests.get("https://data.vatsim.net/v3/vatsim-data.json", timeout=30)
print(r.status_code)
data = r.json()
print("update:", data["general"]["update_timestamp"])
print("pilotos conectados:", len(data["pilots"]))
print("campos de un piloto:", list(data["pilots"][0].keys()))

In [ ]:
print(BRONZE)

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://data.vatsim.net/v3/vatsim-data.json"


def extraer_de_api(desde: str | None = None) -> pd.DataFrame:
    """Extrae el snapshot actual de pilotos conectados a VATSIM.

    VATSIM no tiene histórico ni filtro por fecha: cada llamada es una foto
    del momento. El histórico se construye acumulando snapshots en bronze,
    uno por corrida, cada uno identificado por 'snapshot_ts'.

    Args:
        desde: Watermark de la última corrida (update_timestamp ya ingerido).
            Si el snapshot actual no es más nuevo, se devuelve vacío para no
            reingerir la misma foto.

    Returns:
        DataFrame de pilotos, todo como string, con la columna 'snapshot_ts'.
        flight_plan (objeto anidado) se aplana a columnas flight_plan_*.
    """
    r = requests.get(BASE_URL, timeout=30)
    r.raise_for_status()
    data = r.json()

    snapshot_ts = data["general"]["update_timestamp"]
    if desde and snapshot_ts <= desde:
        print(f"Sin snapshot nuevo (watermark={desde}).")
        return pd.DataFrame()

    df = pd.json_normalize(data["pilots"], sep="_")
    df["snapshot_ts"] = snapshot_ts
    print(f"Filas extraídas: {len(df)} (snapshot {snapshot_ts})")
    return df.astype("string")


nuevos = extraer_de_api(desde=leer_watermark())
print(nuevos.shape); nuevos.head()

In [ ]:
import pandas as pd

def extraer_desde_semilla(desde=None):

    if desde:
        df = df[df['fecha'] > desde]

    return df

print("Semilla disponible para desarrollo offline.")


In [ ]:
%pip install pyarrow

In [ ]:
import uuid
from datetime import datetime, timezone

from deltalake import write_deltalake

CLAVE = ["cid", "snapshot_ts"]


def ingerir(extractor, destino: str = BRONZE) -> int:
    """Ingiere en bronze solo las filas nuevas (idempotente).

    Args:
        extractor: Función que recibe 'desde' y devuelve un DataFrame.
        destino: URI de la tabla Delta de bronze.

    Returns:
        Cantidad de filas escritas (0 si no había novedades).
    """
    wm = leer_watermark(destino)
    nuevos = extractor(desde=wm)
    if nuevos is None or nuevos.empty:
        print(f"Sin novedades (watermark={wm}).")
        return 0

    nuevos = nuevos.drop_duplicates(CLAVE)

    # Segunda barrera: descartar claves que ya existen en bronze.
    try:
        existentes = DeltaTable(destino, storage_options=STORAGE_OPTIONS).to_pandas(columns=CLAVE)
        claves_viejas = set(map(tuple, existentes.values))
        nuevos = nuevos[~nuevos[CLAVE].apply(tuple, axis=1).isin(claves_viejas)]
    except TableNotFoundError:
        pass  # primera corrida

    if nuevos.empty:
        print(f"Todo lo extraído ya estaba en bronze (watermark={wm}).")
        return 0

    nuevos = nuevos.assign(
        _ingested_at=datetime.now(timezone.utc).isoformat(),
        _run_id=uuid.uuid4().hex,
    )
    write_deltalake(destino, nuevos, mode="append", storage_options=STORAGE_OPTIONS)
    print(f"Ingeridas {len(nuevos)} filas (watermark previo={wm}).")
    return len(nuevos)


n = ingerir(extraer_de_api)
print("Filas ingeridas:", n)

In [ ]:
n = ingerir(extraer_de_api)
print("Filas ingeridas:", n)